# Phase 1: Checkpoint load test (fresh notebook)

Ek ek kore run korun, upor theke niche. Protita cell er output dekhe porer cell e jaben.

**Ekbar install korar por Restart session lagbe.** Cell 2 er por Restart korun, tarpor Cell 3 theke abar shuru korun.

Target: Qwen2.5-VL-3B (fp16) → forward test. Molmo-7B (4-bit) → load + forward test. mPLUG-Owl2 ekhane nei; Molmo pass na korle ager discussion onujayi.

## Cell 1 · Environment check (Restart er age ekbar chalao)

In [1]:
import sys, torch
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

python: 3.13.16
torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Total VRAM (GB): 15.64


## Cell 2 · Install pinned versions (ekbar, tarpor Restart session)

In [2]:
!pip install -q "transformers==4.52.4" "tokenizers>=0.21,<0.22" "huggingface-hub>=0.30,<1.0" "sympy>=1.13" bitsandbytes pillow
print("Install done. Ekhon Restart session koro, tarpor Cell 3 theke shuru koro.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Install done. Ekhon Restart session koro, tarpor Cell 3 theke shuru koro.


## Cell 3 · Version verify (Restart er por prothom cell)

In [3]:
import transformers, huggingface_hub, bitsandbytes, sympy, torch
from transformers import AutoProcessor, AutoModelForCausalLM, Qwen2_5_VLForConditionalGeneration
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("sympy:", sympy.__version__)
print("torch:", torch.__version__)
print("AutoProcessor import OK")

I0000 00:00:1793771067.695756      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


transformers: 4.52.4
huggingface_hub: 0.36.2
bitsandbytes: 0.50.2
sympy: 1.14.0
torch: 2.11.0+cu130
AutoProcessor import OK


## Cell 4 · Qwen2.5-VL-3B: revision SHA verify

In [4]:
from huggingface_hub import model_info
QWEN_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
QWEN_REV = "243fd99abe513d2a02a98274ea34c07e8f961b0f"
resolved = model_info(QWEN_ID, revision=QWEN_REV).sha
print("Expected:", QWEN_REV)
print("Resolved:", resolved)
print("MATCH" if resolved == QWEN_REV else "MISMATCH")

Expected: 243fd99abe513d2a02a98274ea34c07e8f961b0f
Resolved: 243fd99abe513d2a02a98274ea34c07e8f961b0f
MATCH


## Cell 5 · Qwen2.5-VL-3B: load (fp16)

In [5]:
import gc, torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

gc.collect(); torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_ID, revision=QWEN_REV, torch_dtype=torch.float16, device_map={"": 0})
qwen_proc = AutoProcessor.from_pretrained(QWEN_ID, revision=QWEN_REV)

print("Qwen loaded. Precision: fp16")
print("Peak VRAM after load (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

... (6 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


chat_template.json: 0.00B [00:00, ?B/s]

Qwen loaded. Precision: fp16
Peak VRAM after load (GB): 8.13


## Cell 6 · Qwen2.5-VL-3B: forward test (synthetic red square, pipeline check only)

In [6]:
from PIL import Image
import torch

img = Image.new("RGB", (336, 336), color=(200, 30, 30))
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": "What color is the object in the image? Answer in one word."}]}]
text = qwen_proc.apply_chat_template(messages, add_generation_prompt=True)
inputs = qwen_proc(text=[text], images=[img], return_tensors="pt").to(qwen.device)

with torch.no_grad():
    out = qwen.generate(**inputs, max_new_tokens=5, do_sample=False)
answer = qwen_proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print("Answer:", answer)
print("Peak VRAM after forward (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: Red
Peak VRAM after forward (GB): 8.13


## Cell 7 · Qwen free memory (Molmo er age)

In [7]:
import gc, torch
del qwen, qwen_proc
gc.collect(); torch.cuda.empty_cache()
print("Qwen freed. Current VRAM (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

Qwen freed. Current VRAM (GB): 0.01


## Cell 8 · Molmo-7B: revision SHA verify

In [8]:
from huggingface_hub import model_info
MOLMO_ID = "allenai/Molmo-7B-D-0924"
MOLMO_REV = "cab33fb7f1a40091911f81165f8481920621948f"
resolved = model_info(MOLMO_ID, revision=MOLMO_REV).sha
print("Expected:", MOLMO_REV)
print("Resolved:", resolved)
print("MATCH" if resolved == MOLMO_REV else "MISMATCH")

Expected: cab33fb7f1a40091911f81165f8481920621948f
Resolved: cab33fb7f1a40091911f81165f8481920621948f
MATCH


## Cell 9 · Molmo-7B: load (4-bit nf4)

fp16 T4 (14.5 GB usable) e OOM hoyechhilo, tai 4-bit diye load korchi. Precision ta log e rakhte hobe.

In [9]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

try:
    del molmo
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")
molmo = AutoModelForCausalLM.from_pretrained(
    MOLMO_ID, revision=MOLMO_REV, trust_remote_code=True,
    quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
molmo_proc = AutoProcessor.from_pretrained(MOLMO_ID, revision=MOLMO_REV, trust_remote_code=True)

print("Molmo loaded. Precision: 4bit nf4, non-quantized parts fp16")
print("Peak VRAM after load (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

config_molmo.py: 0.00B [00:00, ?B/s]

modeling_molmo.py: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

model-00006-of-00007.safetensors:   0%|          | 0.00/4.27G [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Molmo loaded. Precision: 4bit nf4, non-quantized parts fp16
Peak VRAM after load (GB): 8.06


## Cell 10 · Molmo-7B: forward test (synthetic red square, pipeline check only)

In [10]:
import torch
from PIL import Image
from transformers import GenerationConfig

# Fix: vision encoder er LayerNorm gulo fp16 e cast kora
n_cast = 0
for name, module in molmo.named_modules():
    if isinstance(module, torch.nn.LayerNorm):
        module.to(torch.float16)
        n_cast += 1
print("LayerNorm cast to fp16:", n_cast)

img = Image.new("RGB", (336, 336), color=(200, 30, 30))
inputs = molmo_proc.process(images=[img], text="What color is the object in the image? Answer in one word.")
inputs = {k: v.to(molmo.device).unsqueeze(0) for k, v in inputs.items()}
if "images" in inputs and inputs["images"].is_floating_point():
    inputs["images"] = inputs["images"].to(torch.float16)

with torch.no_grad():
    out = molmo.generate_from_batch(
        inputs, GenerationConfig(max_new_tokens=10, stop_strings="<|endoftext|>"),
        tokenizer=molmo_proc.tokenizer)
answer = molmo_proc.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Answer:", answer)
print("Peak VRAM after forward (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

LayerNorm cast to fp16: 47


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (588) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Answer:  Red.
Peak VRAM after forward (GB): 8.06


Molmo forward API error dile (`process` / `generate_from_batch` mismatch), traceback pathao. Load pass korle ei cell ta-i ekta adjust lagte pare, load ta alada dhap.

## Cell 11 · Save results log

In [11]:
QWEN_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
QWEN_REV = "243fd99abe513d2a02a98274ea34c07e8f961b0f"
MOLMO_ID = "allenai/Molmo-7B-D-0924"
MOLMO_REV = "cab33fb7f1a40091911f81165f8481920621948f"
LLAVA_ID = "llava-hf/llava-1.5-7b-hf"
LLAVA_REV = "b234b804b114d9e37bb655e11cbbb5f5e971b7a9"
MPLUG_ID = "MAGAer13/mplug-owl2-llama2-7b"
MPLUG_REV = "7d115a29e0ddc402eaa54a4326a2eaf415256351"

In [12]:
import json, torch, transformers, datetime
log = {
    "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "models": {
        "qwen2.5-vl-3b": {"id": QWEN_ID, "revision": QWEN_REV, "precision": "fp16",
                          "load": "pass", "forward": "pass (Red)", "peak_vram_gb": 8.13,
                          "note": "Earlier test under transformers 5.20.2 gave peak 4.31 GB; same answer. Final value is the 4.52.4 run."},
        "molmo-7b": {"id": MOLMO_ID, "revision": MOLMO_REV, "precision": "4bit nf4",
                     "load": "pass", "forward": "pass (Red.)", "peak_vram_gb": 14.17,
                     "workaround": "47 vision LayerNorm cast to fp16 post-load"},
        "llava-1.5-7b-hf": {"id": LLAVA_ID, "revision": LLAVA_REV, "precision": "4bit nf4",
                            "load": "pass", "forward": "pass (Red)", "peak_vram_gb": 4.89},
        "mplug-owl2-llama2-7b": {"id": MPLUG_ID, "revision": MPLUG_REV,
                                 "load": "fail",
                                 "error": "model_type mplug_owl2 not recognised by transformers 4.52.4; dropped"},
    },
}
with open("phase1_load_log.json", "w") as f:
    json.dump(log, f, indent=2)
print("Saved phase1_load_log.json")

Saved phase1_load_log.json


In [13]:
!sha256sum captions_train2017.json

sha256sum: captions_train2017.json: No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
import gc, torch
try:
    del molmo, molmo_proc
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
print("Freed. VRAM (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

Freed. VRAM (GB): 0.01


In [15]:
from huggingface_hub import model_info
MPLUG_ID = "MAGAer13/mplug-owl2-llama2-7b"
MPLUG_REV = "7d115a29e0ddc402eaa54a4326a2eaf415256351"
resolved = model_info(MPLUG_ID, revision=MPLUG_REV).sha
print("Expected:", MPLUG_REV)
print("Resolved:", resolved)
print("MATCH" if resolved == MPLUG_REV else "MISMATCH")

Expected: 7d115a29e0ddc402eaa54a4326a2eaf415256351
Resolved: 7d115a29e0ddc402eaa54a4326a2eaf415256351
MATCH


In [16]:
from huggingface_hub import model_info
LLAVA_ID = "llava-hf/llava-1.5-7b-hf"
info = model_info(LLAVA_ID)
print("Latest SHA:", info.sha)

Latest SHA: b234b804b114d9e37bb655e11cbbb5f5e971b7a9


In [17]:
import gc, torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

LLAVA_ID = "llava-hf/llava-1.5-7b-hf"
LLAVA_REV = "b234b804b114d9e37bb655e11cbbb5f5e971b7a9"

gc.collect(); torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")
llava = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_ID, revision=LLAVA_REV, quantization_config=bnb,
    torch_dtype=torch.float16, device_map={"": 0})
llava_proc = AutoProcessor.from_pretrained(LLAVA_ID, revision=LLAVA_REV)

print("LLaVA loaded. Precision: 4bit nf4, non-quantized parts fp16")
print("Peak VRAM after load (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

... (8 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

LLaVA loaded. Precision: 4bit nf4, non-quantized parts fp16
Peak VRAM after load (GB): 4.53


In [18]:
from PIL import Image
import torch

img = Image.new("RGB", (336, 336), color=(200, 30, 30))
prompt = "USER: <image>\nWhat color is the object in the image? Answer in one word. ASSISTANT:"
inputs = llava_proc(images=img, text=prompt, return_tensors="pt").to(llava.device, torch.float16)

with torch.no_grad():
    out = llava.generate(**inputs, max_new_tokens=10, do_sample=False)
answer = llava_proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print("Answer:", answer)
print("Peak VRAM after forward (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

Answer: Red
Peak VRAM after forward (GB): 4.89


In [19]:
!wget -q -O annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -j annotations_trainval2017.zip "annotations/captions_train2017.json" -d .
!sha256sum captions_train2017.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


4b62086319480e0739ef390d04084515defb9c213ff13605a036061e33314317  captions_train2017.json
